# Post-Training Data Collection

Notebook for building the 5-stage customer-service post-training dataset.
Stages: cold-start CoT SFT → general SFT → DPO → reward model → GRPO.

In [1]:
# Standard library imports
import json
import os
import random
import re
import time
from pathlib import Path
from pprint import pprint
from typing import Any, Dict, List, Optional, Tuple


In [2]:
# Third-party imports
import jsonlines
import numpy as np
import pandas as pd
import requests
from pydantic import BaseModel, Field, field_validator

print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("jsonlines: imported")
print("requests:", requests.__version__)

pandas: 2.3.3
numpy: 2.2.6
jsonlines: imported
requests: 2.34.2


# Setting up schema for all 5 stages of the post-training pipeline

In [3]:
from pydantic import BaseModel, Field, field_validator
from typing import Any, Dict, List, Optional

## Stage 1: Class to handle data for Cold Start COT 

In [3]:
class ColdStartCoTSFTExample(BaseModel): 
    prompt: str = Field(..., description = "Customer Query") 
    response: str = Field(..., description = "Assistant response with <thinking> reasoning trace")

    @field_validator("response")
    @classmethod
    def must_contain_thinking(cls, v:str) -> str: 
        if "<thinking>" not in v or "</thinking>" not in v:
            raise ValueError("Response must contain <thinking>...</thinking> reasoning trace")
        return v

## Stage 2: Class to handle data for general SFT

In [4]:
class GeneralSFTExample(BaseModel): 
    prompt: str = Field(..., description = "Customer Query")
    response: str = Field(..., description = "Assistant response (no thinking trace required)")

## Stage 3: Class to handle data for DPO - Direct Preference Optimization

In [5]:
class DPOExample(BaseModel): 
    prompt: str = Field(..., description = "Customer Query")
    chosen: str = Field(..., description = "Assistant response (no thinking trace required)")
    rejected: str = Field(..., description = "A flawed response to contrast with chosen")

    @field_validator("chosen", "rejected")
    @classmethod
    def not_empty(cls, v:str) -> str:
        if not v.strip():
            raise ValueError("chosen and rejected must be non-empty")
        return v

## Stage 4: Class to handle data for Reward Model

In [6]:
class Rubric(BaseModel): 
    helpfulness: int = Field(...,ge=1, le=5)
    correctness: int = Field(..., ge=1, le=5)
    tone: int = Field(..., ge=1,le=5)


class RewardModelExample(BaseModel): 
    prompt: str = Field(..., description = "customer query")
    response: str = Field(..., description = "Candidate response to score")
    score: int = Field(..., ge=1, le=5, description = "Overall score between 1 and 5")
    rubric: Optional[Rubric] = Field(None, description = "Breakdown per dimension")

## Stage 5: Class to handle data for GRPO- Group Relative Policy Optimization

In [7]:
class GRPOExample(BaseModel): 
    prompt: str = Field(..., description = "Customer query")
    verifiable_reward: Dict[str,bool] = Field(..., 
                                              description = "Criteria dict with boolean checks, e.g. {'return_window_mentioned': True}")

    @field_validator("verifiable_reward")
    @classmethod
    def at_least_one_criterion(cls, v: Dict[str, bool]) -> Dict[str,bool]: 
        if len(v) == 0: 
            raise ValueError("Must have at least 1 verifiable criterion")
        return v
    

## Validator Script to run over the jsonl file

In [8]:
import jsonlines

def validate_jsonl(path: str, schema: type[BaseModel]) -> list[Dict]: 
    """
    Load and validate every line in the jsonl file against the Pydantic Schema. 
    Returns a list of all validation errors as dict : {line_number: errors}
    """

    errors = []
    with jsonlines.open(path) as reader: 
        for i, obj in enumerate(reader): 
            try: 
                schema(**obj)
            except Exception as e: 
                errors.append({i: str(e)})
    return errors
    

In [16]:
import os 
os.environ["OPENROUTER_API_KEY"] = ""

In [17]:
import json
import requests
import time
# os.environ["OPENROUTER_API_KEY"] = ""
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"


In [18]:
def call_openrouter(model: str, system_prompt: str, user_prompt: str, n: int = 1) -> list[Dict]:
    """Call OpenRouter and return parsed json responses."""
    headers = {
       "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json", 
    }
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        "n": n,
        "response_format": {"type": "json_object"},
    }
    resp = requests.post(OPENROUTER_URL, headers=headers, json=payload, timeout=60)
    resp.raise_for_status()
    choices = resp.json()["choices"]
    results = []
    for choice in choices:
        content = choice["message"]["content"]
        results.append(json.loads(content))
    return results

# Retail Customer Service Domain

In [19]:
DOMAIN = "retail customer service (order status, returns, refunds, promotions)"

STAGE_PROMPTS = {
    "01_cold_start_cot_sft": {
        "model": "deepseek/deepseek-v4-flash",
        "system": f"You generate training data for post-training LLMs in {DOMAIN}. "
                  "Return a JSON object with 'prompt' and 'response' keys. "
                  "The response MUST contain a <thinking>...</thinking> block showing "
                  "step-by-step reasoning (check policy, look up info, decide action), "
                  "followed by the customer-facing answer.",
        "user": "Generate one realistic customer service scenario. "
                "The customer asks about a return or refund. "
                "Show the agent's reasoning inside <thinking> tags, then give the final reply.",
        "schema": ColdStartCoTSFTExample,
    },
    "02_general_sft": {
        "model": "deepseek/deepseek-v4-flash",
        "system": f"You generate training data for post-training LLMs in {DOMAIN}. "
                  "Return a JSON object with 'prompt' and 'response' keys. "
                  "The response is a direct, helpful answer — no thinking trace.",
        "user": "Generate one realistic customer service Q&A about order status or promotions.",
        "schema": GeneralSFTExample,
    },
    "03_dpo": {
        "model": "deepseek/deepseek-v4-flash",
        "system": f"You generate preference training data for {DOMAIN}. "
                  "Return a JSON object with 'prompt', 'chosen', and 'rejected' keys. "
                  "'chosen' is an ideal response. 'rejected' is a flawed response "
                  "(too blunt, wrong policy, hallucinated info, or pushy tone).",
        "user": "Generate one customer service scenario with a good response and a bad response.",
        "schema": DPOExample,
    },
    "04_reward_model": {
        "model": "deepseek/deepseek-v4-flash",
        "system": f"You generate reward model training data for {DOMAIN}. "
                  "Return a JSON object with 'prompt', 'response', 'score' (1-5), "
                  "and 'rubric' (helpfulness, correctness, tone — each 1-5).",
        "user": "Generate one customer service response and score it on a 1-5 rubric.",
        "schema": RewardModelExample,
    },
    "05_grpo": {
        "model": "deepseek/deepseek-v4-flash",
        "system": f"You generate GRPO prompts with verifiable rewards for {DOMAIN}. "
                  "Return a JSON object with 'prompt' and 'verifiable_reward' keys. "
                  "'verifiable_reward' is a dict of boolean criteria that a correct "
                  "response should satisfy.",
        "user": "Generate one customer service prompt where the correct answer can be "
                "verified against specific criteria (e.g., return_window_mentioned, "
                "refund_method_correct, policy_adhered).",
        "schema": GRPOExample,
    },
}

## Generating 2 samples

In [20]:
N_PER_STAGE = 2

for stage_name, config in STAGE_PROMPTS.items():
    print(f"\n=== {stage_name} ===")
    results = call_openrouter(
        model=config["model"],
        system_prompt=config["system"],
        user_prompt=config["user"],
        n=N_PER_STAGE,
    )
    for i, obj in enumerate(results):
        try:
            validated = config["schema"](**obj)
            print(f"\n--- Sample {i+1}: VALID ---")
            print(validated.model_dump_json(indent=2))
        except Exception as e:
            print(f"\n--- Sample {i+1}: INVALID ---")
            print(f"Error: {e}")
            print(f"Raw: {json.dumps(obj, indent=2)}")


=== 01_cold_start_cot_sft ===

--- Sample 1: VALID ---
{
  "prompt": "Hi, I bought a dress from your store last week, but it doesn't fit. Can I return it and get a refund?",
  "response": "<thinking>First, I need to verify the purchase date against our 30-day return policy. The customer says 'last week,' which is within the timeframe. I'll check the order details assuming standard terms: the dress must be unworn with tags, and it’s not a final sale item. Since the issue is fit, no signs of damage or wear, I can proceed. Action: Authorize the return, provide a prepaid label, and explain the refund process (5-7 business days after receipt). Ensure clear instructions for the customer.</thinking>Yes, you can absolutely return the dress for a refund! Since you purchased it last week, it’s within our 30-day return window. Please make sure the dress is unworn with all tags still attached. I’ll email you a prepaid return label shortly. Once we receive and inspect it, your refund will be issue

# Now generating the 500 samples plus holdout eval set

In [21]:
import random
from typing import Any

In [23]:
POLICY = (
    "30-day return window from delivery date. "
    "Refunds go to the original payment method within 5-7 business days. "
    "Damaged or wrong items are always refundable with a free return label, "
    "even outside the 30-day window."
)

TOPICS = [
    "return-in-window", 
    "return-out-of-window",
    "refund-not-received", 
    "damaged-item",
    "wrong-item-shipped", 
    "late-delivery", 
    "coupon-not-applying",
    "cancel-order", 
    "exchange-size",
    "price-adjustment",
]

TONES = ["polite", "frustrated", "confused", "urgent"]

ITEMS = [
    "vitamins",
    "running sneakers",
    "skincare serum",
    "wireless headphones",
    "protein powder",
    "kids pajamas",
    "electric toothbrush",
    "coffee maker",
    "yoga mat",
    "phone case",
]

PAYMENT_METHODS = ["Visa", "Mastercard", "PayPal", "gift card"]


In [25]:
def build_spec(rng: random.Random | None = None) -> dict[str, Any]: 
    """Build one diverse retail CS scenario seed."""
    r = rng or random
    return {
         "topic": r.choice(TOPICS),
        "tone": r.choice(TONES),
        "item": r.choice(ITEMS),
        "days_since_purchase": r.randint(1, 60),
        "payment_method": r.choice(PAYMENT_METHODS),
        "order_id": f"ORD-{r.randint(100000, 999999)}"
    }

In [26]:
#checks
rng = random.Random(42)
specs_500 = [build_spec(rng) for _ in range(500)]

print("Sample specs:")
for s in specs_500[:3]:
    print(s)

Sample specs:
{'topic': 'return-out-of-window', 'tone': 'polite', 'item': 'protein powder', 'days_since_purchase': 16, 'payment_method': 'Mastercard', 'order_id': 'ORD-246316'}
{'topic': 'return-out-of-window', 'tone': 'polite', 'item': 'phone case', 'days_since_purchase': 28, 'payment_method': 'Visa', 'order_id': 'ORD-131244'}
{'topic': 'return-out-of-window', 'tone': 'frustrated', 'item': 'wireless headphones', 'days_since_purchase': 33, 'payment_method': 'Visa', 'order_id': 'ORD-688508'}


In [27]:
from collections import Counter
print("\nTopic distribution:")
for topic, n in sorted(Counter(s["topic"] for s in specs_500).items()):
    print(f"  {topic}: {n}")



Topic distribution:
  cancel-order: 45
  coupon-not-applying: 55
  damaged-item: 59
  exchange-size: 48
  late-delivery: 54
  price-adjustment: 39
  refund-not-received: 42
  return-in-window: 51
  return-out-of-window: 46
  wrong-item-shipped: 61


In [28]:
print(f"\nPOLICY:\n{POLICY}")
print(f"Total specs: {len(specs_500)}")


POLICY:
30-day return window from delivery date. Refunds go to the original payment method within 5-7 business days. Damaged or wrong items are always refundable with a free return label, even outside the 30-day window.
Total specs: 500


# Building User prompt now

In [30]:
def build_user_prompt(spec: Dict) -> str: 
    """Inject one scenario spec into the stage-01 user prompt."""
    return (
        "Generate one realistic customer service scenario.\n"
        f"Customer topic: {spec['topic']}.\n"
        f"Customer tone: {spec['tone']}.\n"
        f"Item purchased: {spec['item']}, {spec['days_since_purchase']} days ago.\n"
        f"Paid with: {spec['payment_method']} (order {spec['order_id']}).\n"
        f"Store policy: {POLICY}\n"
        "The customer asks about a return or refund. "
        "Show the agent's reasoning inside <thinking> tags "
        "(check policy, apply facts, decide action), then give the final reply."
    )

In [32]:
# checks 
rng = random.Random(7)

In [33]:
for s in [build_spec(rng) for _ in range(2)]:
    print(build_user_prompt(s))
    print("\n" + "-" * 60 + "\n")

Generate one realistic customer service scenario.
Customer topic: late-delivery.
Customer tone: frustrated.
Item purchased: electric toothbrush, 42 days ago.
Paid with: Visa (order ORD-175954).
Store policy: 30-day return window from delivery date. Refunds go to the original payment method within 5-7 business days. Damaged or wrong items are always refundable with a free return label, even outside the 30-day window.
The customer asks about a return or refund. Show the agent's reasoning inside <thinking> tags (check policy, apply facts, decide action), then give the final reply.

------------------------------------------------------------

Generate one realistic customer service scenario.
Customer topic: exchange-size.
Customer tone: polite.
Item purchased: kids pajamas, 38 days ago.
Paid with: Visa (order ORD-632084).
Store policy: 30-day return window from delivery date. Refunds go to the original payment method within 5-7 business days. Damaged or wrong items are always refundable w

In [34]:
import json 
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

In [35]:
OUT_PATH = "../data/01_cold_start_cot_sft/data.jsonl"
FAILURE_PATH = "../data/01_cold_start_cot_sft/failures.jsonl"
CONFIG = STAGE_PROMPTS["01_cold_start_cot_sft"]

In [36]:
def generate_one(spec: dict) -> dict:
    """Generate + validate one sample. Returns record or None on final failure."""
    for attempt in range(3):
        try:
            objs = call_openrouter(
                model=CONFIG["model"],
                system_prompt=CONFIG["system"],
                user_prompt=build_user_prompt(spec),
                n=1,
            )
            record = CONFIG["schema"](**objs[0])
            return {"spec": spec, **record.model_dump()}
        except Exception as e:
            if attempt == 2:
                return {"spec": spec, "error": str(e)}
            time.sleep(2**attempt)
    return None

In [37]:
def generate_dataset(specs: list[dict], out_path: str = OUT_PATH,
                     failures_path: str = FAILURE_PATH, workers: int = 8) -> None:
    """Generate specs[skip:] (resume by line count), append valid records as they arrive."""
    already = sum(1 for _ in open(out_path)) if os.path.exists(out_path) else 0
    todo = specs[already:]
    print(f"Resuming: {already} done, {len(todo)} to go")

    done, ok, failed = already, 0, 0
    with open(out_path, "a") as fout, open(failures_path, "a") as ferr:
        with ThreadPoolExecutor(max_workers=workers) as pool:
            futures = {pool.submit(generate_one, s): s for s in todo}
            for fut in as_completed(futures):
                res = fut.result()
                if res is None or "error" in res:
                    failed += 1
                    ferr.write(json.dumps(res) + "\n")
                else:
                    ok += 1
                    fout.write(json.dumps(res) + "\n")
                done += 1
                if done % 50 == 0:
                    print(f"  {done}/{already + len(todo)} done, {failed} failed")
    print(f"Done. {ok} valid, {failed} failed")


In [ ]:
import os
generate_dataset(specs_500)

Resuming: 0 done, 500 to go
  50/500 done, 0 failed
  100/500 done, 0 failed


# Coding Domain Problem Exploration

In [2]:
%pip install datasets

  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached async_timeout-5.0.1-py3-none-any.whl.metadata (5.1 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 14.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.8/719.8 kB 20.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 25.1 MB/s  0:00:00
Using cached async_timeout-5.0.1-py3-none-any.whl (6.2 kB)
Using cached aiosignal-1.4.0-py3-none-any.whl (7.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.1/35.1 MB 31.6 MB/s  0:00:012.2 MB/s eta 0:00:0102
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25/25 [datasets] 24/25 [datasets]ggingface-hub]
Note: you may need to restart the kernel to use updated packages.


In [5]:
%pip install --upgrade datasets huggingface_hub

Note: you may need to restart the kernel to use updated packages.


In [15]:
from datasets import load_dataset, disable_progress_bar
import json

disable_progress_bar()
ds = load_dataset("google-research-datasets/mbpp", split="train[:5]")

example = ds[0]
print("=== Raw fields ===")
for k, v in example.items():
    print(f"{k}: {repr(v)[:120]}")

def to_sft_example(ex):
    tests = "\n".join(ex["test_list"])
    instruction = (
        f"Write a Python function to solve the following problem.\n\n"
        f"Problem: {ex['text']}\n\n"
        f"Your code must pass these tests:\n{tests}"
    )
    return {"messages": [
        {"role": "user", "content": instruction},
        {"role": "assistant", "content": ex["code"]},
    ]}

sft = to_sft_example(example)
print("\n=== SFT format (chat-style) ===")
print(json.dumps(sft, indent=2))

=== Raw fields ===
task_id: 601
text: 'Write a function to find the longest chain which can be formed from the given set of pairs.'
code: 'class Pair(object): \r\n\tdef __init__(self, a, b): \r\n\t\tself.a = a \r\n\t\tself.b = b \r\ndef max_chain_length(arr,
test_list: ['assert max_chain_length([Pair(5, 24), Pair(15, 25),Pair(27, 40), Pair(50, 60)], 4) == 3', 'assert max_chain_length([Pa
test_setup_code: ''
challenge_test_list: []

=== SFT format (chat-style) ===
{
  "messages": [
    {
      "role": "user",
      "content": "Write a Python function to solve the following problem.\n\nProblem: Write a function to find the longest chain which can be formed from the given set of pairs.\n\nYour code must pass these tests:\nassert max_chain_length([Pair(5, 24), Pair(15, 25),Pair(27, 40), Pair(50, 60)], 4) == 3\nassert max_chain_length([Pair(1, 2), Pair(3, 4),Pair(5, 6), Pair(7, 8)], 4) == 4\nassert max_chain_length([Pair(19, 10), Pair(11, 12),Pair(13, 14), Pair(15, 16), Pair(31, 54)], 5) ==

In [25]:
from pathlib import Path
BASE = Path("data/coding")
for stage in ["01_cold_start_cot_sft", "02_general_sft", "03_dpo", 
              "04_reward_model", "05_grpo"]:
    (BASE / stage).mkdir(parents=True, exist_ok=True)

In [23]:
# Loading mbpp again
from datasets import load_dataset, disable_progress_bar
disable_progress_bar()
mbpp = load_dataset("google-research-datasets/mbpp", split="train")
# Sample 20 per stage (use different seeds or allow overlap)
import random
random.seed(42)
indices = random.sample(range(len(mbpp)), 20)
samples = [mbpp[i] for i in indices]

/Users/unmeshmali/Downloads/Unmesh/deepmlhub/projects/llm-retard-lab/courses/deeplearning_posttraining/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [27]:
samples[2]

{'task_id': 613,
 'text': 'Write a function to find the maximum value in record list as tuple attribute in the given tuple list.',
 'code': 'def maximum_value(test_list):\r\n  res = [(key, max(lst)) for key, lst in test_list]\r\n  return (res) ',
 'test_list': ["assert maximum_value([('key1', [3, 4, 5]), ('key2', [1, 4, 2]), ('key3', [9, 3])]) == [('key1', 5), ('key2', 4), ('key3', 9)]",
  "assert maximum_value([('key1', [4, 5, 6]), ('key2', [2, 5, 3]), ('key3', [10, 4])]) == [('key1', 6), ('key2', 5), ('key3', 10)]",
  "assert maximum_value([('key1', [5, 6, 7]), ('key2', [3, 6, 4]), ('key3', [11, 5])]) == [('key1', 7), ('key2', 6), ('key3', 11)]"],
 'test_setup_code': '',
 'challenge_test_list': []}

In [25]:
samples[2]['text']

'Write a function to find the maximum value in record list as tuple attribute in the given tuple list.'

In [26]:
samples[2]['code']

'def maximum_value(test_list):\r\n  res = [(key, max(lst)) for key, lst in test_list]\r\n  return (res) '

In [38]:
for key,value in samples[2].items(): 
    print(key)
    print(value)

task_id
613
text
Write a function to find the maximum value in record list as tuple attribute in the given tuple list.
code
def maximum_value(test_list):
  res = [(key, max(lst)) for key, lst in test_list]
  return (res) 
test_list
["assert maximum_value([('key1', [3, 4, 5]), ('key2', [1, 4, 2]), ('key3', [9, 3])]) == [('key1', 5), ('key2', 4), ('key3', 9)]", "assert maximum_value([('key1', [4, 5, 6]), ('key2', [2, 5, 3]), ('key3', [10, 4])]) == [('key1', 6), ('key2', 5), ('key3', 10)]", "assert maximum_value([('key1', [5, 6, 7]), ('key2', [3, 6, 4]), ('key3', [11, 5])]) == [('key1', 7), ('key2', 6), ('key3', 11)]"]
test_setup_code

challenge_test_list
[]


# Helper functions

In [14]:
def call_openrouter_text(model: str, system_prompt: str, user_prompt: str) -> str:
    """Call OpenRouter in text mode (no JSON constraint). Returns raw string."""
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        "n": 1,
    }
    resp = requests.post(OPENROUTER_URL, headers=headers, json=payload, timeout=60)
    resp.raise_for_status()
    return resp.json()["choices"][0]["message"]["content"]


In [15]:
def format_coding_prompt(text: str, test_list: list[str]) -> str: 
    """
    We need a consistent coding prompt. Coding prompt will include problem statement and tests to run. 
    The coding prompt is a string. But the test to run is a list of strings. 
    """
    tests = "\n".join(test_list)
    return (
        f"Write a Python function to solve the following problem.\n\n"
        f"Problem: {text}\n\n"
        f"Your code must pass these tests:\n{tests}"
    )
    

In [16]:
def normalize_code(code : str) -> str: 
    """Convert Windows \\r\\n to Unix \\n for clean storage."""
    return code.replace("\r\n", "\n").replace("\r", "\n")

In [17]:
def write_jsonl(path: str, records: list[dict]):
    """Write a list of dicts to a JSONL file, creating dirs as needed."""
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with jsonlines.open(path, "w") as writer: 
        for record in records:
            writer.write(record)

## Stage 1: Cold Start SFT data collection

In [76]:
# Pattern: Model generates reasoning inside <thinking> tags, then code. Prompt comes from MBPP (problem + test cases). Response saved as raw text.
# SYSTEM_COT = (
#     "You are a Python tutor generating training data.\n"
#     "Given a coding problem and test cases:\n"
#     "1. Think step-by-step inside <thinking>...</thinking> tags "
#     "(algorithm choice, edge cases, complexity)\n"
#     "2. Output the complete Python solution after the thinking block.\n"
#     "Do NOT wrap in JSON. Just output thinking + code."
# )
# The above system prompt was too weak for the model to output the thinking traces. It kept failing at schema validation for the first stage


SYSTEM_COT_JSON = (
    "You are a Python tutor generating training data.\n"
    "Given a coding problem and test cases, return valid JSON with exactly 2 keys:\n"
    '- "reasoning": step-by-step thinking about algorithm choice, edge cases, complexity\n'
    '- "code": the complete Python solution\n\n'
    "IMPORTANT: Escape all special characters in the code string for valid JSON "
    "(especially backslashes and quotes)."
)


In [77]:
records = []
N = len(samples)

In [78]:
for i, ex in enumerate(samples): 
    prompt = format_coding_prompt(ex["text"], ex["test_list"])
    try:
        results = call_openrouter(
            model="deepseek/deepseek-v4-flash",
            system_prompt=SYSTEM_COT_JSON,
            user_prompt=f"Problem and tests:\n\n{prompt}",
            n=1,
        )
        obj = results[0]
        response = f"<thinking>{obj['reasoning']}</thinking>\n\n{obj['code']}"
        record = {"prompt": prompt, "response": response}
        ColdStartCoTSFTExample(**record)
        records.append(record)
    except Exception as e:
        print(f"  Sample {i}: INVALID — {e}")
    time.sleep(0.5)
    print(f"  {i+1}/{N} done", end="\r")

path = str(BASE / "01_cold_start_cot_sft" / "data.jsonl")
write_jsonl(path, records)
print(f"\nStage 1: {len(records)} valid / {N} total -> {path}")
    

  20/20 done
Stage 1: 20 valid / 20 total -> data/coding/01_cold_start_cot_sft/data.jsonl


In [81]:
records[0]["prompt"]

"Write a Python function to solve the following problem.\n\nProblem: Write a function to convert a date of yyyy-mm-dd format to dd-mm-yyyy format.\n\nYour code must pass these tests:\nassert change_date_format('2026-01-02')=='02-01-2026'\nassert change_date_format('2021-01-04')=='04-01-2021'\nassert change_date_format('2030-06-06')=='06-06-2030'"

In [82]:
records[0]["response"]

'<thinking>The problem requires converting a date from \'yyyy-mm-dd\' to \'dd-mm-yyyy\' format. The simplest approach is to split the input string by the hyphens into a list of three strings: year, month, day. Then, concatenate them in reverse order with hyphens: day, month, year. This is O(1) time and space because the input length is fixed. Edge cases: The input is assumed to always be in the correct format, so no validation is needed. The function will return the formatted string as required.</thinking>\n\ndef change_date_format(date_str):\n    parts = date_str.split(\'-\')\n    return f"{parts[2]}-{parts[1]}-{parts[0]}"'

## Stage 2: General SFT (0 API Calls,, MBPP ground truth) 

In [26]:
records = []

for i, ex in enumerate(samples):
    prompt = format_coding_prompt(ex["text"], ex["test_list"])
    code = normalize_code(ex["code"])
    record = {"prompt": prompt, "response": code}
    try: 
        GeneralSFTExample(**record)
        records.append(record) 
    except Exception as e: 
        print(f"Sample {i}: INVALID - {e}")
path = str(BASE / "02_general_sft" / "data.jsonl") 
write_jsonl(path, records) 
print(f"Stage 2 : {len(records)} valid / {len(samples)} totla -> {path}")

Stage 2 : 20 valid / 20 totla -> data/coding/02_general_sft/data.jsonl


In [28]:
records[0]

{'prompt': "Write a Python function to solve the following problem.\n\nProblem: Write a function to convert a date of yyyy-mm-dd format to dd-mm-yyyy format.\n\nYour code must pass these tests:\nassert change_date_format('2026-01-02')=='02-01-2026'\nassert change_date_format('2021-01-04')=='04-01-2021'\nassert change_date_format('2030-06-06')=='06-06-2030'",
 'response': "import re\ndef change_date_format(dt):\n        return re.sub(r'(\\d{4})-(\\d{1,2})-(\\d{1,2})', '\\\\3-\\\\2-\\\\1', dt)\n        return change_date_format(dt)"}

## Stage 3: DPO (will need 20 API calls) 

In [29]:
SYSTEM_FLAWED = (
        "You are generating preference training data.\n"
    "Write a SUBTLY FLAWED Python solution for the given problem.\n"
    "It should look plausible but contain a logical bug, miss an edge case, "
    "or use an inefficient approach.\n"
    "Do NOT add comments like 'this is flawed'. Make it look like a real attempt.\n"
    "Just output the code. No thinking trace."

)

In [31]:
N = len(samples)

In [32]:
records = []
for i,ex in enumerate(samples): 
    prompt = format_coding_prompt(ex["text"], ex["test_list"])
    chosen = normalize_code(ex["code"])
    try: 
        rejected = call_openrouter_text("deepseek/deepseek-v4-flash", SYSTEM_FLAWED, prompt)
        record = {"prompt": prompt, "chosen": chosen, "rejected": rejected}
        DPOExample(**record)
        records.append(record) 
    except Exception as e: 
        print(f"Sample {i} fails with {e}")
    time.sleep(0.5) 
    print(f"{i+1}/ {N} done")

path = str(BASE / "03_dpo" / "data.jsonl")
write_jsonl(path, records)
print(f"\nStage 3: {len(records)} valid / {len(samples)} total -> {path}")

1/ 20 done
2/ 20 done
3/ 20 done
4/ 20 done
5/ 20 done
6/ 20 done
7/ 20 done
8/ 20 done
9/ 20 done
10/ 20 done
11/ 20 done
12/ 20 done
13/ 20 done
14/ 20 done
15/ 20 done
16/ 20 done
17/ 20 done
18/ 20 done
19/ 20 done
20/ 20 done

Stage 3: 20 valid / 20 total -> data/coding/03_dpo/data.jsonl


In [36]:
records[1]['prompt']

'Write a Python function to solve the following problem.\n\nProblem: Write a function to find the item with maximum occurrences in a given list.\n\nYour code must pass these tests:\nassert max_occurrences([2,3,8,4,7,9,8,2,6,5,1,6,1,2,3,4,6,9,1,2])==2\nassert max_occurrences([1, 3,5, 7,1, 3,13, 15, 17,5, 7,9,1, 11])==1\nassert max_occurrences([1, 2, 3,2, 4, 5,1, 1, 1])==1'

In [40]:
records[1]['chosen']

'def max_occurrences(list1):\n    max_val = 0\n    result = list1[0] \n    for i in list1:\n        occu = list1.count(i)\n        if occu > max_val:\n            max_val = occu\n            result = i \n    return result'

In [41]:
records[1]['rejected']

'\n```python\ndef max_occurrences(items):\n    counts = {}\n    for item in items:\n        counts[item] = counts.get(item, 0) + 1\n    \n    max_count = 0\n    max_item = None\n    for item, count in counts.items():\n        if count > max_count:\n            max_count = count\n            max_item = item\n        elif count == max_count:\n            # Keep the smaller item when counts tie\n            if item < max_item:\n                max_item = item\n    \n    if max_item is None and items:\n        # Fallback for empty list? But list is non-empty in tests\n        max_item = items[0]\n    return max_item\n```'

## Stage 4: Reward Model Training Data Collection

In [42]:
SYSTEM_GENERATE = (
        "Write a Python solution for the given problem and test cases.\n"
    "Just output the code. No thinking trace."
)

In [45]:
SYSTEM_SCORE = (
    "You are a code quality judge. Score the given Python solution on 3 dimensions "
    "(each 1-5) plus an overall score (1-5).\n\n"
    "Return ONLY valid JSON with these exact keys:\n"
    '- "score": integer 1-5\n'
    '- "rubric": {"helpfulness": 1-5, "correctness": 1-5, "tone": 1-5}\n\n'
    "Example: {\"score\": 4, \"rubric\": {\"helpfulness\": 4, \"correctness\": 3, \"tone\": 5}}"
)

In [46]:
records = []
for i, ex in enumerate(samples): 
    prompt = format_coding_prompt(ex["text"], ex["test_list"])
    try: 
        code = call_openrouter_text("deepseek/deepseek-v4-flash", SYSTEM_GENERATE, prompt)
        score_raw = call_openrouter_text("deepseek/deepseek-v4-flash", SYSTEM_SCORE, f"Problme:\n {prompt} \n\n Solution:\n {code}")
        match = re.search(r"\{.*\}", score_raw, re.DOTALL)
        score_data = json.loads(match.group())
        record = {
            "prompt": prompt,
            "response": code,
            "score": score_data["score"],
            "rubric": score_data["rubric"],
        }
        RewardModelExample(**record)
        records.append(record)
    except Exception as e: 
        print(f"  Sample {i}: INVALID — {e}")
    time.sleep(0.5)
    print(f"  {i+1}/{N} done", end="\r")

path = str(BASE / "04_reward_model" / "data.jsonl")
write_jsonl(path, records)
print(f"\nStage 4: {len(records)} valid / {len(samples)} total -> {path}")

  Sample 13: INVALID — 1 validation error for RewardModelExample
response
  Input should be a valid string [type=string_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type
  20/20 done
Stage 4: 19 valid / 20 total -> data/coding/04_reward_model/data.jsonl


In [47]:
records[1]

{'prompt': 'Write a Python function to solve the following problem.\n\nProblem: Write a function to find the item with maximum occurrences in a given list.\n\nYour code must pass these tests:\nassert max_occurrences([2,3,8,4,7,9,8,2,6,5,1,6,1,2,3,4,6,9,1,2])==2\nassert max_occurrences([1, 3,5, 7,1, 3,13, 15, 17,5, 7,9,1, 11])==1\nassert max_occurrences([1, 2, 3,2, 4, 5,1, 1, 1])==1',
 'response': "Here's a Python solution for the problem:  \n\n```python\ndef max_occurrences(lst):\n    if not lst:\n        return None\n    freq = {}\n    for item in lst:\n        freq[item] = freq.get(item, 0) + 1\n    max_count = max(freq.values())\n    return max(item for item, count in freq.items() if count == max_count)\n```",
 'score': 5,
 'rubric': {'helpfulness': 5, 'correctness': 5, 'tone': 5}}

In [51]:
records[1].keys()

dict_keys(['prompt', 'response', 'score', 'rubric'])

In [52]:
records[1]['prompt']

'Write a Python function to solve the following problem.\n\nProblem: Write a function to find the item with maximum occurrences in a given list.\n\nYour code must pass these tests:\nassert max_occurrences([2,3,8,4,7,9,8,2,6,5,1,6,1,2,3,4,6,9,1,2])==2\nassert max_occurrences([1, 3,5, 7,1, 3,13, 15, 17,5, 7,9,1, 11])==1\nassert max_occurrences([1, 2, 3,2, 4, 5,1, 1, 1])==1'

In [53]:
records[1]['response']

"Here's a Python solution for the problem:  \n\n```python\ndef max_occurrences(lst):\n    if not lst:\n        return None\n    freq = {}\n    for item in lst:\n        freq[item] = freq.get(item, 0) + 1\n    max_count = max(freq.values())\n    return max(item for item, count in freq.items() if count == max_count)\n```"

In [54]:
records[1]['score']

5

In [55]:
records[1]['rubric']

{'helpfulness': 5, 'correctness': 5, 'tone': 5}

## Stage 5: GRPO (prompt and verifiable rewards) 

In [58]:
records = []

In [60]:
for i, ex in enumerate(samples): 
    prompt = format_coding_prompt(ex["text"], ex["test_list"])
    criteria = {}
    for j, test in enumerate(ex["test_list"]): 
        name = test.replace("assert ", "passes_")[:60]
        criteria[name] = True
    record = {"prompt": prompt, "verifiable_reward": criteria}
    try: 
        GRPOExample(**record)
        records.append(record)
    except Exception as e: 
        print(f"  Sample {i}: INVALID — {e}")

path = str(BASE / "05_grpo" / "data.jsonl")
write_jsonl(path, records)
print(f"Stage 5: {len(records)} valid / {len(samples)} total -> {path}")

Stage 5: 20 valid / 20 total -> data/coding/05_grpo/data.jsonl


## 5: Final Validation - run Pydantic checks on all 5 jsonL files

In [62]:
VALIDATION_SCHEMAS = {
    "01_cold_start_cot_sft": ColdStartCoTSFTExample,
    "02_general_sft": GeneralSFTExample,
    "03_dpo": DPOExample,
    "04_reward_model": RewardModelExample,
    "05_grpo": GRPOExample,
}

In [63]:
for stage, schema in VALIDATION_SCHEMAS.items():
    path = str(BASE / stage / "data.jsonl")
    line_count = sum(1 for _ in jsonlines.open(path))
    errors = validate_jsonl(path, schema)
    status = "✅" if not errors else "❌"
    print(f"{status} {stage}: {line_count} lines, {len(errors)} errors")

✅ 01_cold_start_cot_sft: 20 lines, 0 errors
✅ 02_general_sft: 20 lines, 0 errors
✅ 03_dpo: 20 lines, 0 errors
✅ 04_reward_model: 19 lines, 0 errors
✅ 05_grpo: 20 lines, 0 errors


# ----------------------

# Data Collection for "Tool Use" Training

# ----------------------

In [98]:
# Tools - model sees this tool list - this is rendered into every prompt

TOOLS = [
    {"name": "get_order_status",
    "description": "Look up the current status and ETA of an order",
    "parameters" : {
        "type": "object", 
        "properties": {
            "order_id": {"type":  "string", "description": "Order ID like 'A12345'"}
        },
        "required": ["order_id"]
    },
    }, 
    {"name": "initiate_return",
    "description": "Start a return for an order",
    "parameters" : {
        "type": "object", 
        "properties": {
            "order_id": {"type":"string"},
            "reason" : {"type": "string", 
                       "description": "One of: 'damaged', 'wrong_item', 'changed_mind'"},
        },
        "required" : ["order_id", "reason"],
    },
    },
    {
        "name": "search_products",
        "description": "Search the product catalog",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string"},
                "max_price": {"type": "number", "description": "Optional price ceiling USD"},
            },
            "required": ["query"],
        },
    },
    {
        "name": "apply_coupon",
        "description": "Apply a coupon code to a cart total, returns discounted total",
        "parameters": {
            "type": "object",
            "properties": {
                "coupon_code": {"type": "string"},
                "cart_total": {"type": "number"},
            },
            "required": ["coupon_code", "cart_total"],
        },
    },
]

In [99]:
# The state: what the VERIFIER uses (never shown to the model)
ORDERS_DB = {
    "A12345": {"status": "shipped", "eta": "2026-07-22", "items": ["running shoes"]},
    "B67890": {"status": "processing", "eta": None, "items": ["water bottle"]},
    "C11223": {"status": "delivered", "eta": None, "items": ["yoga mat"]},
}

PRODUCTS_DB = [
    {"name": "running shoes", "price": 79.99},
    {"name": "water bottle", "price": 18.50},
    {"name": "yoga mat", "price": 32.00},
    {"name": "resistance bands", "price": 24.99},
]

COUPONS_DB = {
    "SAVE10": {"pct_off": 10, "min_total": 25.0},
    "FIT20":  {"pct_off": 20, "min_total": 50.0},
}

In [100]:
def get_order_status(order_id: str):
    return ORDERS_DB.get(order_id, {"error": "order_not_found"})

In [101]:
def initiate_return(order_id: str, reason : str): 
    if order_id not in ORDERS_DB:
        return {"error": "order_not_found"}
    return {"return_id": f"R-{order_id}", "reason": reason, "status": "approved"}

In [102]:
def search_products(query: str, max_price: float = None):
    hits = [p for p in PRODUCTS_DB if query.lower() in p["name"]]
    if max_price is not None:
        hits = [p for p in hits if p["price"] <= max_price]
    return {"results": hits}

In [103]:
def apply_coupon(coupon_code: str, catr_total: str): 
    c = COUPONS_DB.get(coupon_code)
    if not c:
        return {"error": "invalid_coupon"}
    if cart_total < c["min_total"]:
        return {"error": "min_total_not_met", "min_total": c["min_total"]}
    return {"discounted_total": round(cart_total * (1 - c["pct_off"] / 100), 2)}

In [104]:
TOOL_IMPLS = {
    "get_order_status": get_order_status,
    "initiate_return": initiate_return,
    "search_products": search_products,
    "apply_coupon": apply_coupon,
}

In [105]:
def execute_tool(call: dict) -> dict:
    """
    Execute a model produced call. Never raises. Errors are data. 
    Design decision 1: 
    at GRPO time you'll execute garbage model outputs and need a score, not a crash. Every failure model returns as error dict
    """
    if not isinstance(call, dict) or "name" not in call or "arguments" not in call: 
        return {"error": "malformed_call"}
    fn = TOOL_IMPLS.get(call["name"])
    if fn is None: 
         return {"error": "unknown_tool"}
    try: 
        return fn(**call["arguments"])
    except TypeError as e: 
        return {"error": f"bad_arguments: {e}"} 
    

## The generator (samples the DB to author verified pairs)

In [106]:
TOOL_LIST_STR = json.dumps(TOOLS, indent=2)

In [107]:
def format_tool_prompt(user_query: str) -> str:
    """
    One prompt format used for all 5 stages. 
    If stage 1 and stage 5 render the rejistry differently, you're training
    and evaluating on different distributions.
    """
    return (
        "You are a retail shopping assistant with access to these tools:\n"
        f"{TOOL_LIST_STR}\n\n"
        'Respond with a single tool call as raw JSON: {"name": ..., "arguments": {...}}\n'
        "If no tool is needed, answer directly.\n\n"
        f"Customer: {user_query}"
    )

In [108]:
REASON_PHRASES = {
    "damaged": "it arrived damaged",
    "wrong_item": "you sent the wrong item",
    "changed mind": "I changed my mind", 
}

In [109]:
def gen_order_status(): 
    oid  = random.choice(list(ORDERS_DB))
    query = random.choice([
        f"Where is my order {oid}?",
        f"Has order {oid} shipped yet?",
        f"What's the status of order {oid}?",
    ])
    return query, {"name": "get_order_status", "arguments": {"order_id": oid}}

In [110]:
def gen_return():
    oid = random.choice(list(ORDERS_DB))
    reason = random.choice(list(REASON_PHRASES))
    query = random.choice([
        f"I need to return order {oid}, {REASON_PHRASES[reason]}",
        f"Start a return for {oid} — {REASON_PHRASES[reason]}",
    ])
    return query, {"name": "initiate_return", "arguments": {"order_id": oid, "reason": reason}}

def gen_apply_coupon():
    code = random.choice(list(COUPONS_DB))
    total = random.choice([19.99, 34.50, 62.00, 85.25])
    return (f"Can you apply {code} to my ${total} cart?",
            {"name": "apply_coupon", "arguments": {"coupon_code": code, "cart_total": total}})

def gen_search():
    term = random.choice(["shoes", "bottle", "yoga", "bands"])
    return (f"Do you sell {term}?",
            {"name": "search_products", "arguments": {"query": term}})

GENERATORS = [gen_order_status, gen_return, gen_apply_coupon, gen_search]

In [111]:
def sample_tool_example() -> dict:
    """Design decision #3: expected_call is KNOWN BY CONSTRUCTION.
    No LLM needed to create stage 2 / stage 5 data — the environment is the author."""
    query, call = random.choice(GENERATORS)()
    return {"query": query, "prompt": format_tool_prompt(query), "expected_call": call}

In [112]:
for _ in range(6):
    ex = sample_tool_example()
    obs = execute_tool(ex["expected_call"])
    print(ex["query"])
    print("  call:", ex["expected_call"])
    print("  obs: ", obs, "\n")

Do you sell shoes?
  call: {'name': 'search_products', 'arguments': {'query': 'shoes'}}
  obs:  {'results': [{'name': 'running shoes', 'price': 79.99}]} 

Can you apply FIT20 to my $62.0 cart?
  call: {'name': 'apply_coupon', 'arguments': {'coupon_code': 'FIT20', 'cart_total': 62.0}}
  obs:  {'error': "bad_arguments: apply_coupon() got an unexpected keyword argument 'cart_total'"} 

Has order C11223 shipped yet?
  call: {'name': 'get_order_status', 'arguments': {'order_id': 'C11223'}}
  obs:  {'status': 'delivered', 'eta': None, 'items': ['yoga mat']} 

Where is my order B67890?
  call: {'name': 'get_order_status', 'arguments': {'order_id': 'B67890'}}
  obs:  {'status': 'processing', 'eta': None, 'items': ['water bottle']} 

Can you apply FIT20 to my $34.5 cart?
  call: {'name': 'apply_coupon', 'arguments': {'coupon_code': 'FIT20', 'cart_total': 34.5}}
  obs:  {'error': "bad_arguments: apply_coupon() got an unexpected keyword argument 'cart_total'"} 

What's the status of order A12345?

In [113]:
print(execute_tool({"name": "get_order_status", "arguments": {}}))                 # missing arg
print(execute_tool({"name": "get_order_stats", "arguments": {"order_id": "A1"}}))  # wrong tool
print(execute_tool("not even a dict")) 

{'error': "bad_arguments: get_order_status() missing 1 required positional argument: 'order_id'"}
{'error': 'unknown_tool'}
{'error': 'malformed_call'}
